# Etapa 3 — Análise Exploratória (EDA)

**Objetivo:** entender quem são os 50 mil clientes da amostra e **como a posse de produtos varia com idade, renda e segmento**.
Isso orienta as features da segmentação (Etapa 4) e do recomendador (Etapa 5).

Toda a lógica está em `src/eda.py`; este notebook só orquestra e interpreta. As figuras são salvas em `docs/figures/` para os slides.

> Execute de cima para baixo após rodar `python -m src.prepare_data`.

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Image, display
from src import config, eda

# Variáveis de ambiente permitem apontar para outra base sem editar o notebook
DATA_PATH = Path(os.environ.get("EDA_DATA_PATH", config.PROCESSED_FILE))
FIG_DIR = Path(os.environ.get("EDA_FIG_DIR", config.FIGURES_DIR))

eda.set_style()
pd.set_option("display.max_columns", 50)
df = eda.load_processed(DATA_PATH)

## 1. Visão geral da base

In [ ]:
print(f"{len(df):,} clientes únicos · {df.shape[1]} colunas · snapshot {df[config.DATE_COL].iloc[0]:%Y-%m-%d}")
df[["age", "renta", "antiguedad", "n_produtos"]].describe().round(1)

In [ ]:
# Composição categórica
pd.concat(
    {c: df[c].value_counts(normalize=True).mul(100).round(1) for c in ["segmento", "faixa_etaria", "ind_actividad_cliente"]},
    axis=0,
).to_frame("% clientes")

## 2. Distribuições numéricas

**O que observar:**
- **Idade:** a literatura sobre este dataset descreve uma distribuição **bimodal**, com um pico de universitários (~20–25) e outro de adultos (~40–50). *Confirmar no gráfico.*
- **Renda:** tem cauda longa à direita (por isso usamos escala log e quintis). O pico artificial na mediana é efeito da imputação.
- **Antiguidade:** contas antigas indicam clientes mais fiéis.

In [ ]:
fig = eda.fig_distributions(df); eda.save_fig(fig, "01_distribuicoes", FIG_DIR)

## 3. Segmentos

In [ ]:
fig = eda.fig_segments(df); eda.save_fig(fig, "02_segmentos", FIG_DIR)

**Leitura esperada:** o segmento *universitario* deve concentrar os jovens, e *top* deve ter a maior renda.
Se isso se confirmar, `segmento` e `age` carregam informação parecida. Vale lembrar disso ao interpretar os clusters.

## 4. Posse de produtos

In [ ]:
fig = eda.fig_product_ownership(df); eda.save_fig(fig, "03_posse_produtos", FIG_DIR)

In [ ]:
eda.ownership_rate(df).round(2)

In [ ]:
fig = eda.fig_products_per_client(df); eda.save_fig(fig, "04_produtos_por_cliente", FIG_DIR)

**Por que isso importa para o modelo:** a posse de produtos é **muito desbalanceada**. Poucos produtos (conta corrente, por exemplo) dominam, e vários têm < 1% de posse.
Consequências:
- na Etapa 5, a **AUC por produto** é mais informativa que a acurácia (que seria alta só por prever "não possui");
- produtos raros (derivativos, por exemplo) terão estimativas instáveis, o que justifica **agrupar em classes** para a carteira.

## 5. Posse × perfil (idade, renda, segmento)

Cada painel tem **sua própria escala**. Liquidez fica perto de 100% e renda variável em poucos %; em um eixo único, as classes pequenas sumiriam.

In [ ]:
for by, name in [("faixa_etaria", "05_classe_por_idade"), ("renta_quintil", "06_classe_por_renda"), ("segmento", "07_classe_por_segmento")]:
    fig = eda.fig_penetration_by(df, by); eda.save_fig(fig, name, FIG_DIR)

In [ ]:
eda.group_penetration_by(df, "renta_quintil")

In [ ]:
fig = eda.fig_heatmap_by(df, "faixa_etaria"); eda.save_fig(fig, "08_heatmap_idade", FIG_DIR)

In [ ]:
fig = eda.fig_heatmap_by(df, "renta_quintil"); eda.save_fig(fig, "09_heatmap_renda", FIG_DIR)

## 6. Correlação entre produtos

Usamos o **coeficiente phi** (Pearson aplicado a variáveis 0/1). Pares fortemente correlacionados indicam produtos que "andam juntos".
Exemplo esperado: *conta salário × folha de pagamento × pensão*. Isso tem duas consequências:
- o recomendador pode sugerir o "próximo produto natural";
- na segmentação, contar produtos **por classe** evita que um mesmo comportamento seja contado três vezes.

In [ ]:
fig = eda.fig_correlation(df); eda.save_fig(fig, "10_correlacao_produtos", FIG_DIR)

In [ ]:
eda.top_correlated_pairs(df, 10)

## 7. Insights para os slides

As frases abaixo são **geradas a partir dos dados** (os números não foram digitados à mão), então continuam corretas se a amostra mudar.

In [ ]:
for i, txt in enumerate(eda.generate_insights(df), 1):
    print(f"{i}. {txt}")

### Como transformar em narrativa (preencher após rodar)

| # | Mensagem para a banca | Evidência (figura) |
|---|---|---|
| 1 | A maioria dos clientes só usa produtos de conta: há espaço para um assessor | 03, insight 3 |
| 2 | Investir cresce com renda e idade, então o perfil demográfico **é** preditivo | 05, 06 |
| 3 | Produtos formam "pacotes" (alta correlação), e o recomendador explora isso | 10 |
| 4 | Dados desbalanceados levam a usar métricas por produto (AUC, MAP@7) | 03, 04 |
| 5 | Limitação: renda imputada e dados espanhóis de 2016 | insight final |

## 8. Ponte para a Etapa 4 (segmentação)

Features que a EDA justifica: `age`, `log(renta)`, `antiguedad`, `ind_actividad_cliente`, `segmento` (one-hot) e **`n_<classe>`** (contagem por classe, já criada em `src/features.py`).
Ficam de fora: `sexo` (decisão D11) e os 24 produtos individuais (redundantes com as contagens, e esparsos demais para o K-Means).